In [7]:
import json
import pickle
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.preprocessing import LabelEncoder


In [5]:
with open("intents.json") as file:
    data  = json.load(file)

traning_sentences = []
traning_labels = []
labels = []
responses = []

for intent in data['intents']:
    for patern in intent['patterns']:
        traning_sentences.append(patern)
        traning_labels.append(intent['tag'])
    responses.append(intent['responses'])

    if intent['tag'] not in labels:
        labels.append(intent['tag'])
number_of_classes = len(labels)


In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(traning_labels)
traning_labels = label_encoder.transform(traning_labels)

vocab_size =1000
max_len = 20
embedding_dim = 16
ovv_token = "<OVV"

tokenizer = Tokenizer(num_words=vocab_size, oov_token = ovv_token)
tokenizer.fit_on_texts(traning_sentences)
sequences = tokenizer.texts_to_sequences(traning_sentences)
padded_sequence = pad_sequences(sequences,truncating='post',maxlen =max_len)
 


In [ ]:
model = Sequential()
model.add(Embedding(vocab_size,embedding_dim,input_legth = max_len))
model.add(GlobalAveragePooling1D())
model.add(Dense(16,activation='relu'))
model.add(Dense(16,activation='relu'))
model.add(Dense(number_of_classes, activation="softmax"))

model.compile(loss = 'sparse_catogorial_crossentropy',optimizer = "adam",matrics =  ["accuracy"])

model.summary()

model.fit(padded_sequence,np.array(traning_labels),epoch = 100)


In [ ]:
model.save("chat_model.h5")

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f, protocol=pickle.HIGHEST_PROTOCOL)

with open("label_encoder.pkl", "wb") as encoder_file:
    pickle.dump(label_encoder, encoder_file, protocol=pickle.HIGHEST_PROTOCOL)